In [1]:
from __future__ import annotations
import os, random, io, zipfile, gzip
from typing import List, Optional, Iterable
import pandas as pd

path='20250915-0000-gleif-goldencopy-lei2-golden-copy.csv'

# Minimal column set you actually use downstream (tweak as needed)
GLEIF_MIN_COLS = [
    "LEI",
    "Entity.LegalName",
    "Entity.LegalAddress.AddressLine1","Entity.LegalAddress.AddressLine2","Entity.LegalAddress.AddressLine3",
    "Entity.LegalAddress.City","Entity.LegalAddress.Region","Entity.LegalAddress.PostalCode","Entity.LegalAddress.Country",
    "Entity.HeadquartersAddress.AddressLine1","Entity.HeadquartersAddress.AddressLine2","Entity.HeadquartersAddress.AddressLine3",
    "Entity.HeadquartersAddress.City","Entity.HeadquartersAddress.Region","Entity.HeadquartersAddress.PostalCode","Entity.HeadquartersAddress.Country",
]

def _open_csv_for_pandas(path: str) -> dict:
    """
    Returns kwargs for pandas.read_csv that handle .gz/.zip without loading to RAM.
    """
    if path.lower().endswith(".gz"):
        return {"filepath_or_buffer": path, "compression": "gzip"}
    if path.lower().endswith(".zip"):
        # Pandas can read a single-file zip by path directly.
        # If the zip has multiple CSVs, pandas reads the first one.
        return {"filepath_or_buffer": path, "compression": "zip"}
    return {"filepath_or_buffer": path}

def read_gleif_reservoir(
    path: str,
    usecols: [str] = None,
    reservoir_size: int = 250_000,
    chunksize: int = 200_000,
) -> pd.DataFrame:
    """
    Stream a huge GLEIF CSV/CSV.GZ/ZIP and keep only a uniform random sample
    of at most `reservoir_size` rows in memory. Returns a pandas DataFrame of the sample.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    # Make sure we only parse strings (no giant Python objects); pandas' native
    # 'string' dtype is more memory friendly than 'object' when combined with pyarrow backend.
    dtype = None  # let pandas read as object, we’ll convert at the end (fastest for wide text)
    read_kwargs = dict(_open_csv_for_pandas(path),
                       usecols=usecols,
                       dtype=dtype,
                       keep_default_na=False,
                       chunksize=chunksize,
                       low_memory=True,
                       memory_map=True)

    reservoir = []   # list[tuple]
    cols = None
    seen = 0

    for chunk in pd.read_csv(**read_kwargs):
        if cols is None:
            cols = list(chunk.columns)

        # Iterate as tuples (no per-row Series allocations)
        for row in chunk.itertuples(index=False, name=None):
            seen += 1
            if len(reservoir) < reservoir_size:
                reservoir.append(row)
            else:
                j = random.randint(1, seen)
                if j <= reservoir_size:
                    reservoir[j - 1] = row

    if not reservoir:
        return pd.DataFrame(columns=usecols or cols or [])

    out = pd.DataFrame.from_records(reservoir, columns=cols)

    # Downcast/optimize: use pandas' nullable string (pyarrow backend if available)
    try:
        # pandas >= 2 supports dtype_backend='pyarrow' for more compact memory
        out = out.convert_dtypes(dtype_backend="pyarrow")
    except Exception:
        # fallback: at least move to pandas' string dtype
        for c in out.columns:
            if out[c].dtype == object:
                out[c] = out[c].astype("string")

    return out


# ---------- Optional: one-off creation of a compact Parquet ----------
# Read once in chunks, write an on-disk parquet that loads much faster/lighter later.
def build_minimal_parquet(
    path: str,
    out_parquet: str,
    usecols: [str] = None,
    chunksize: int = 200_000,
):
    import pyarrow as pa
    import pyarrow.parquet as pq

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    read_kwargs = dict(_open_csv_for_pandas(path),
                       usecols=usecols,
                       dtype=None,
                       keep_default_na=False,
                       chunksize=chunksize,
                       low_memory=True,
                       memory_map=True)

    writer = None
    try:
        for chunk in pd.read_csv(**read_kwargs):
            # Convert chunk to Arrow table using types that compress well
            try:
                chunk = chunk.convert_dtypes(dtype_backend="pyarrow")
            except Exception:
                for c in chunk.columns:
                    if chunk[c].dtype == object:
                        chunk[c] = chunk[c].astype("string")
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(out_parquet, table.schema, compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()


In [2]:
df = pd.read_csv(path)

C:\Users\AleksandarKovacevic\AppData\Local\Temp\ipykernel_46816\3280352130.py:1: DtypeWarning: Columns (6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,35,36,37,38,39,40,42,44,47,48,49,50,51,52,54,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,75,76,77,78,79,80,81,82,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,188,189,192,194,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,220,226,227,228,229,230,231,232,233,234,235,236,239,245,246,247,248,249,250,251,252,253,254,255,256,257,258,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,298,299,300,301,302,303,304,305,306,307,308,309,310,312,320,321,322,323,324,325,326,327,328,330,331,333,334,336) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [3]:
df.columns.tolist()

['LEI',
 'Entity.LegalName',
 'Entity.LegalName.xmllang',
 'Entity.OtherEntityNames.OtherEntityName.1',
 'Entity.OtherEntityNames.OtherEntityName.1.xmllang',
 'Entity.OtherEntityNames.OtherEntityName.1.type',
 'Entity.OtherEntityNames.OtherEntityName.2',
 'Entity.OtherEntityNames.OtherEntityName.2.xmllang',
 'Entity.OtherEntityNames.OtherEntityName.2.type',
 'Entity.OtherEntityNames.OtherEntityName.3',
 'Entity.OtherEntityNames.OtherEntityName.3.xmllang',
 'Entity.OtherEntityNames.OtherEntityName.3.type',
 'Entity.OtherEntityNames.OtherEntityName.4',
 'Entity.OtherEntityNames.OtherEntityName.4.xmllang',
 'Entity.OtherEntityNames.OtherEntityName.4.type',
 'Entity.OtherEntityNames.OtherEntityName.5',
 'Entity.OtherEntityNames.OtherEntityName.5.xmllang',
 'Entity.OtherEntityNames.OtherEntityName.5.type',
 'Entity.TransliteratedOtherEntityNames.TransliteratedOtherEntityName.1',
 'Entity.TransliteratedOtherEntityNames.TransliteratedOtherEntityName.1.xmllang',
 'Entity.TransliteratedOtherEnt

In [3]:
df['Entity.EntityStatus'].unique()

array(['ACTIVE', 'INACTIVE', nan], dtype=object)

In [7]:
df_small=df[['LEI',
 'Entity.LegalName',
 'Entity.LegalAddress.FirstAddressLine',
 'Entity.LegalAddress.AddressNumber',
 'Entity.LegalAddress.AddressNumberWithinBuilding',
 'Entity.LegalAddress.AdditionalAddressLine.1',
 'Entity.LegalAddress.AdditionalAddressLine.2',
 'Entity.LegalAddress.AdditionalAddressLine.3',
 'Entity.LegalAddress.City',
 'Entity.LegalAddress.Region',
 'Entity.LegalAddress.Country',
 'Entity.LegalAddress.PostalCode',
 'Entity.RegistrationAuthority.RegistrationAuthorityID',
 'Entity.RegistrationAuthority.OtherRegistrationAuthorityID',
 'Entity.RegistrationAuthority.RegistrationAuthorityEntityID',
 'Entity.LegalForm.EntityLegalFormCode',
 'Entity.EntityStatus',
 'Registration.InitialRegistrationDate',
 'Registration.RegistrationStatus']][df['Entity.LegalAddress.City'].str.lower().isin(["new york","los angeles","belgrade","vienna","zurich","geneva","london","paris","munich"])]
df_small.to_csv('gleif.small.cityfiltered.csv')

In [10]:
df_small=df_small[df_small['Entity.EntityStatus']=='ACTIVE']

In [11]:
df_small_10k=df_small.sample(n=10000)
df_small_10k.to_csv('gleif.small.cityfiltered10k.csv')